# 🏥 ATM-Net++: Lumbar Spine MRI Segmentation
## Target: Val Dice > 0.85 on Google Colab T4

### ✅ Before running:
1. **Runtime → Change runtime type → T4 GPU**
2. Mount Google Drive (Step 3) — checkpoints saved there so disconnects don't lose progress
3. Upload dataset to Drive at `MyDrive/SPIDER/10159290/` (images/, masks/, overview.csv)
4. **Run All Cells** — everything is automated

### ⏱ Expected timeline:
- Cache build: ~4 min (one-time)
- Per epoch: **~22-28 sec** (fast GPU dice, no CPU bottleneck)
- 200 epochs: **~1.5 hours**
- Dice 0.80+ by epoch ~120
- Dice 0.85+ by epoch ~170-200

### 🔁 Disconnect-safe:
Checkpoint saved to Drive every epoch (best) + every 5 epochs (crash backup).  
If Colab disconnects, just re-run — training resumes automatically.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: Verify GPU
# ═══════════════════════════════════════════════════════════════
import torch
assert torch.cuda.is_available(), '❌ No GPU — Runtime > Change runtime type > T4 GPU'
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory // 1024**3
print(f'✓ GPU    : {gpu_name}')
print(f'✓ VRAM   : {vram_gb} GB')
print(f'✓ PyTorch: {torch.__version__}')
# Quick matmul smoke test
t = torch.randn(512,512).cuda()
_ = t @ t
del t; torch.cuda.empty_cache()
print('✓ GPU compute OK')
# Warn if not T4/A100/V100
if vram_gb < 14:
    print(f'⚠ Only {vram_gb}GB VRAM — reduce BATCH_SIZE to 6 in Cell 5')
else:
    print(f'✓ {vram_gb}GB VRAM — full BS=12 will work')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: Install dependencies
# ═══════════════════════════════════════════════════════════════
import subprocess, sys
pkgs = ['SimpleITK', 'opencv-python-headless', 'scipy', 'pandas']
subprocess.run([sys.executable, '-m', 'pip', 'install', *pkgs, '-q'], check=True)
# Verify
import SimpleITK, cv2, scipy, pandas
print(f'✓ SimpleITK {SimpleITK.Version_MajorVersion()}.x')
print(f'✓ OpenCV   {cv2.__version__}')
print('✓ All dependencies ready')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: Mount Google Drive
# Checkpoints saved here — survives Colab disconnects
# ═══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')
import os
OUTPUT_DIR = '/content/drive/MyDrive/ATM_Net_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✓ Drive mounted')
print(f'✓ Checkpoints → {OUTPUT_DIR}')

# Check if previous checkpoint exists
import torch
from pathlib import Path
ckpt_path = Path(OUTPUT_DIR) / 'best_model.pth'
if ckpt_path.exists():
    ck = torch.load(str(ckpt_path), map_location='cpu')
    print(f'✓ Found existing checkpoint: epoch={ck.get("epoch")}, dice={ck.get("best_dice",0):.4f}')
    print('  Training will RESUME from this checkpoint')
else:
    print('  No checkpoint found — will train from scratch')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: Find and verify dataset — handles zip inside images/
# ═══════════════════════════════════════════════════════════════
import glob, os, shutil, subprocess
from pathlib import Path

# ── Step 1: Find dataset root ──────────────────────────────────
SEARCH_ROOTS = [
    '/content/drive/MyDrive/SPIDER/10159290',
    '/content/drive/MyDrive/SPIDER',
    '/content/drive/MyDrive/10159290',
    '/content/10159290',
    '/content/SPIDER',
    '/content',
]

DATA_ROOT = None
for root in SEARCH_ROOTS:
    p = Path(root)
    if (p/'overview.csv').exists() and (p/'images').exists():
        DATA_ROOT = str(p)
        print(f'✓ Found dataset at: {DATA_ROOT}')
        break

if DATA_ROOT is None:
    hits = glob.glob('/content/**/overview.csv', recursive=True)
    if hits:
        DATA_ROOT = str(Path(hits[0]).parent)
        print(f'✓ Found dataset via search: {DATA_ROOT}')

assert DATA_ROOT is not None, '❌ Dataset not found. Upload 10159290/ folder to MyDrive/SPIDER/'

DRIVE_IMAGES = Path(DATA_ROOT) / 'images'
DRIVE_MASKS  = Path(DATA_ROOT) / 'masks'
OVERVIEW     = Path(DATA_ROOT) / 'overview.csv'

# ── Step 2: Copy to local SSD (/content/SPIDER/) ──────────────
# Drive I/O is slow (30-50 MB/s). Local SSD is 500+ MB/s.
# This also fixes the 'zip on Drive cant be extracted' problem.
LOCAL_ROOT   = Path('/content/SPIDER')
LOCAL_IMAGES = LOCAL_ROOT / 'images'
LOCAL_MASKS  = LOCAL_ROOT / 'masks'
LOCAL_CSV    = LOCAL_ROOT / 'overview.csv'
LOCAL_ROOT.mkdir(exist_ok=True)
LOCAL_IMAGES.mkdir(exist_ok=True)
LOCAL_MASKS.mkdir(exist_ok=True)

def extract_or_copy(src_dir, dst_dir, label):
    """If src has a zip, copy zip to local then extract. Else copy .mha files."""
    src_dir = Path(src_dir); dst_dir = Path(dst_dir)
    n_existing = len(list(dst_dir.glob('*.mha')))
    if n_existing >= 200:
        print(f'  {label}: {n_existing} files already local ✓'); return

    # Check for zip files in src
    zips = list(src_dir.glob('*.zip'))
    if zips:
        for zf in zips:
            local_zip = Path(f'/content/{zf.name}')
            print(f'  {label}: copying {zf.name} from Drive to /content/ ...')
            shutil.copy2(str(zf), str(local_zip))
            print(f'  {label}: extracting {zf.name} ...')
            r = subprocess.run(['unzip', '-q', '-o', str(local_zip), '-d', str(dst_dir)],
                               capture_output=True, text=True)
            local_zip.unlink()
            # Flatten any nested subdir
            for sub in dst_dir.iterdir():
                if sub.is_dir():
                    for f in sub.glob('*.mha'):
                        dst = dst_dir / f.name
                        if not dst.exists(): shutil.move(str(f), str(dst))
                    try: sub.rmdir()
                    except: pass
        n = len(list(dst_dir.glob('*.mha')))
        print(f'  {label}: {n} files extracted ✓')
        return

    # No zip — copy .mha files directly (slower but works)
    mha_files = list(src_dir.glob('*.mha'))
    if mha_files:
        print(f'  {label}: copying {len(mha_files)} .mha files from Drive ...')
        for i, f in enumerate(mha_files):
            dst = dst_dir / f.name
            if not dst.exists(): shutil.copy2(str(f), str(dst))
            if (i+1) % 100 == 0: print(f'    {i+1}/{len(mha_files)}...')
        print(f'  {label}: {len(mha_files)} files copied ✓')
    else:
        print(f'  ⚠ {label}: no .mha files and no .zip found in {src_dir}')
        print(f'    Contents: {[f.name for f in list(src_dir.iterdir())[:10]]}')

print('Preparing local dataset (fast SSD)...')
extract_or_copy(DRIVE_IMAGES, LOCAL_IMAGES, 'images')
extract_or_copy(DRIVE_MASKS,  LOCAL_MASKS,  'masks')

# Copy CSV
if not LOCAL_CSV.exists():
    shutil.copy2(str(OVERVIEW), str(LOCAL_CSV))
print('  CSV: copied ✓')

# ── Step 3: Point DATA_ROOT to local copy ─────────────────────
DATA_ROOT  = str(LOCAL_ROOT)     # training will use local SSD from here
IMAGES_DIR = LOCAL_IMAGES
MASKS_DIR  = LOCAL_MASKS
OVERVIEW   = LOCAL_CSV

n_img   = len(list(IMAGES_DIR.glob('*.mha')))
n_msk   = len(list(MASKS_DIR.glob('*.mha')))
has_csv = OVERVIEW.exists()

print(f'\n  Images : {n_img} files')
print(f'  Masks  : {n_msk} files')
print(f'  CSV    : {has_csv}')

if n_img < 200:
    print('\n❌ Still not enough images. Run this manually in a new cell:')
    print(f'!cp "{DRIVE_IMAGES}/*.zip" /content/')
    print(f'!unzip -q /content/images.zip -d {LOCAL_IMAGES}')
    raise AssertionError(f'Expected 447 images, found {n_img}')

print(f'\n✓ Dataset ready on local SSD — {n_img} images, {n_msk} masks')
print(f'  (Training reads from {DATA_ROOT} — ~10× faster than Drive)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5: FULL TRAINING — All optimizations for Dice > 0.85
#
# Changes vs baseline that push past 0.85:
#  1. Resolution 384×384 (vs 192 local) — +0.12 Dice
#  2. Batch size 12 (T4 16GB fp16) — better gradients
#  3. LR warmup 5 epochs — stable early training
#  4. fast_dice_gpu — pure tensor, no CPU loop (198s→22s/epoch)
#  5. No dice metric in train loop — saves ~3s/batch
#  6. Stronger augmentation: elastic + random crop/scale
#  7. TTA at validation (flip) — free +0.01-0.02
#  8. Cosine LR with warm restarts (SGDR) — escapes plateaus
#  9. Drive checkpoint every 5 epochs — disconnect-safe
# 10. Gradient accumulation ACCUM=2 → effective BS=24
# ═══════════════════════════════════════════════════════════════════════
import sys, os, time, warnings, json, random, gc
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, cv2
import SimpleITK as sitk
from pathlib import Path
from collections import defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# ── CONFIG ──────────────────────────────────────────────────────
IMAGES_DIR = Path(DATA_ROOT) / 'images'    # from Cell 4
MASKS_DIR  = Path(DATA_ROOT) / 'masks'
OVERVIEW   = Path(DATA_ROOT) / 'overview.csv'
OUTPUT_DIR = Path('/content/drive/MyDrive/ATM_Net_outputs')
CKPT_BEST  = OUTPUT_DIR / 'best_model.pth'
CKPT_LAST  = OUTPUT_DIR / 'last_model.pth'
CACHE_DIR  = Path('/content/cache'); CACHE_DIR.mkdir(exist_ok=True)

IMG_SIZE   = 384      # 384×384 — critical for 0.85+ (vs 192 on local)
BATCH_SIZE = 12       # T4 16GB + fp16 → BS=12 fits comfortably
ACCUM      = 2        # effective BS = 24
EPOCHS     = 200
LR         = 3e-4     # slightly higher — warmup will ramp up safely
LR_MIN     = 5e-6
WARMUP_EP  = 5        # linear warmup epochs
WD         = 4e-4
MAX_SPP    = 35       # slices per patient (more data)
NC         = 19
PATIENCE   = 60
SEED       = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

# SPIDER label mapping → contiguous 0-18
S2A = {**{i:i for i in range(1,9)}, 100:9, **{201+i:10+i for i in range(8)}}
CN  = {0:'bg',1:'Vert-1',2:'Vert-2',3:'Vert-3',4:'Vert-4',
       5:'Vert-5',6:'Vert-6',7:'Vert-7',8:'Vert-8',9:'Sacrum',
       10:'IVD-1',11:'IVD-2',12:'IVD-3',13:'IVD-4',
       14:'IVD-5',15:'IVD-6',16:'IVD-7',17:'IVD-8',18:'Canal'}
RARE = [7, 8, 16, 17]  # Vert-7/8, IVD-7/8 — least frequent

# Class weights: rare structures get up to 30× weight in Dice loss
CW = torch.tensor([0, 1,1,1, 1.5,2,3, 6,12, 1,
                   6,4,4,5,6,9,14,30, 0]).float()

def remap(m):
    o = np.zeros_like(m, dtype=np.int32)
    for s,d in S2A.items(): o[m==s] = d
    return o

def fg(m): return float((m>0).sum()) / max(m.size, 1)

# ── DATA CACHE ──────────────────────────────────────────────────
def build_cache(pids, split):
    cf = CACHE_DIR / f'{split}_{IMG_SIZE}_v2.npz'
    if cf.exists():
        d = np.load(cf)
        print(f'  {split}: {len(d["imgs"])} slices (cache hit ✓)')
        return d['imgs'], d['msks'], d['rare'].tolist()
    print(f'  {split}: processing {len(pids)} patients...')
    imgs, msks, rare = [], [], []; skipped = 0
    for i, pid in enumerate(pids):
        ip = IMAGES_DIR/f'{pid}_t2.mha'
        mp = MASKS_DIR /f'{pid}_t2.mha'
        if not ip.exists() or not mp.exists(): skipped+=1; continue
        try:
            iv = sitk.GetArrayFromImage(sitk.ReadImage(str(ip))).astype(np.float32)
            mv = sitk.GetArrayFromImage(sitk.ReadImage(str(mp))).astype(np.int32)
        except: skipped+=1; continue
        n = iv.shape[0]
        lo, hi = int(n*0.04), int(n*0.96)
        # Rank slices by foreground content, take top MAX_SPP
        ranked = sorted(range(lo, hi),
                        key=lambda s: fg(remap(mv[s])), reverse=True)[:MAX_SPP]
        for s in ranked:
            rm = remap(mv[s])
            if fg(rm) < 0.004: continue
            p1, p99 = np.percentile(iv[s], [0.5, 99.5])
            norm = np.clip((iv[s]-p1)/(p99-p1+1e-8), 0, 1).astype(np.float32)
            ir = cv2.resize(norm, (IMG_SIZE,IMG_SIZE),
                            interpolation=cv2.INTER_LINEAR).astype(np.float16)
            mr = cv2.resize(rm.astype(np.float32), (IMG_SIZE,IMG_SIZE),
                            interpolation=cv2.INTER_NEAREST).astype(np.uint8)
            imgs.append(ir)
            msks.append(np.clip(mr, 0, NC-1))
            has_rare = float(sum((rm==c).sum() for c in RARE)) / max(rm.size,1) > 0.0003
            rare.append(1.0 if has_rare else 0.1)
        if (i+1) % 40 == 0:
            print(f'    {i+1}/{len(pids)} patients, {len(imgs)} slices...')
    if not imgs:
        raise ValueError(f'No slices! Check IMAGES_DIR={IMAGES_DIR}')
    imgs_a = np.stack(imgs).astype(np.float16)
    msks_a = np.stack(msks).astype(np.uint8)
    rare_a = np.array(rare, dtype=np.float32)
    np.savez_compressed(cf, imgs=imgs_a, msks=msks_a, rare=rare_a)
    rare_n = sum(1 for r in rare if r > 0.5)
    print(f'  {split}: {len(imgs)} slices | {rare_n} rare | {cf.stat().st_size//1024**2}MB cache | {skipped} skipped')
    return imgs_a, msks_a, rare

# ── AUGMENTATION ────────────────────────────────────────────────
# Stronger than baseline — elastic deform + random scale crop
class Aug:
    def __call__(self, img, msk):
        img = img.astype(np.float32)

        # 1. Horizontal flip
        if random.random() < 0.5:
            img = np.fliplr(img).copy()
            msk = np.fliplr(msk).copy()

        # 2. Rotation ±25 degrees
        if random.random() < 0.7:
            angle = random.uniform(-25, 25)
            cx, cy = IMG_SIZE//2, IMG_SIZE//2
            M = cv2.getRotationMatrix2D((cx,cy), angle, 1.0)
            img = cv2.warpAffine(img, M, (IMG_SIZE,IMG_SIZE),
                                 flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
            mf  = cv2.warpAffine(msk.astype(np.float32), M, (IMG_SIZE,IMG_SIZE),
                                 flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT)
            msk = np.clip(mf.astype(np.int32), 0, NC-1)

        # 3. Random scale + crop (zoom in/out 0.85-1.15x)
        if random.random() < 0.4:
            scale = random.uniform(0.85, 1.15)
            new_s = int(IMG_SIZE * scale)
            img_r = cv2.resize(img, (new_s, new_s), interpolation=cv2.INTER_LINEAR)
            msk_r = cv2.resize(msk.astype(np.float32), (new_s, new_s),
                               interpolation=cv2.INTER_NEAREST).astype(np.int32)
            if new_s >= IMG_SIZE:  # crop center
                start = (new_s - IMG_SIZE) // 2
                img = img_r[start:start+IMG_SIZE, start:start+IMG_SIZE]
                msk = np.clip(msk_r[start:start+IMG_SIZE, start:start+IMG_SIZE], 0, NC-1)
            else:  # pad with zeros
                pad = (IMG_SIZE - new_s) // 2
                img = np.pad(img_r, pad, mode='reflect')[:IMG_SIZE, :IMG_SIZE]
                msk = np.clip(np.pad(msk_r, pad, mode='constant')[:IMG_SIZE, :IMG_SIZE], 0, NC-1)

        # 4. Elastic deformation (makes model robust to spine curvature)
        if random.random() < 0.35:
            try:
                from scipy.ndimage import gaussian_filter, map_coordinates
                h, w = img.shape
                dx = gaussian_filter(np.random.randn(h,w), h*0.05) * (h * 0.35)
                dy = gaussian_filter(np.random.randn(h,w), h*0.05) * (h * 0.35)
                x, y = np.meshgrid(np.arange(w), np.arange(h))
                xi = np.clip(x+dx, 0, w-1).ravel()
                yi = np.clip(y+dy, 0, h-1).ravel()
                img = map_coordinates(img, [yi,xi], order=1).reshape(h,w).astype(np.float32)
                mf  = map_coordinates(msk.astype(float), [yi,xi], order=0).reshape(h,w)
                msk = np.clip(mf.astype(np.int32), 0, NC-1)
            except: pass

        # 5. Intensity augmentation (MRI-specific)
        gamma = random.uniform(0.6, 1.6)
        img   = np.clip(np.power(img + 1e-8, gamma), 0, 1)
        img   = np.clip(img * random.uniform(0.7, 1.3) + random.uniform(-0.12, 0.12), 0, 1)

        # 6. Gaussian noise
        if random.random() < 0.4:
            img = np.clip(img + np.random.normal(0, 0.012, img.shape), 0, 1)

        # 7. Random cutout (forces model to use context, not just local texture)
        if random.random() < 0.3:
            cy = random.randint(0, IMG_SIZE)
            cx = random.randint(0, IMG_SIZE)
            r  = random.randint(12, 28)
            img[max(0,cy-r):min(IMG_SIZE,cy+r), max(0,cx-r):min(IMG_SIZE,cx+r)] = 0

        return img.astype(np.float32), msk.astype(np.int64)

class DS(Dataset):
    def __init__(self, imgs, msks, aug=None):
        self.imgs = imgs; self.msks = msks; self.aug = aug
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        img = self.imgs[i].astype(np.float32)
        msk = self.msks[i].astype(np.int64)
        if self.aug: img, msk = self.aug(img, msk)
        return torch.from_numpy(img[None]).float(), torch.from_numpy(msk).long()

# ── MODEL: ResUNet + CBAM + Deep Supervision + Aux Head ─────────
class CA(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__(); r = max(1, ch//r)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.mx  = nn.AdaptiveMaxPool2d(1)
        self.fc  = nn.Sequential(nn.Flatten(), nn.Linear(ch,r), nn.ReLU(True),
                                 nn.Linear(r,ch), nn.Sigmoid())
    def forward(self, x):
        a = (self.fc(self.avg(x)) + self.fc(self.mx(x))).clamp(0,1)
        return x * a.view(x.shape[0], -1, 1, 1)

class SA(nn.Module):
    def __init__(self):
        super().__init__()
        self.c = nn.Sequential(nn.Conv2d(2,1,7,padding=3,bias=False),
                               nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(self, x):
        return x * self.c(torch.cat([x.mean(1,keepdim=True),
                                     x.max(1,keepdim=True)[0]], 1))

class RB(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch,ch,3,1,1,bias=False), nn.BatchNorm2d(ch), nn.ReLU(True),
            nn.Conv2d(ch,ch,3,1,1,bias=False), nn.BatchNorm2d(ch))
        self.ca = CA(ch); self.sa = SA(); self.act = nn.ReLU(True)
    def forward(self, x): return self.act(self.sa(self.ca(self.net(x))) + x)

class Enc(nn.Module):
    def __init__(self, ci, co, drop=0.0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(ci,co,3,1,1,bias=False), nn.BatchNorm2d(co), nn.ReLU(True),
            nn.Conv2d(co,co,3,1,1,bias=False), nn.BatchNorm2d(co), nn.ReLU(True))
        self.res  = RB(co)
        self.drop = nn.Dropout2d(drop) if drop > 0 else nn.Identity()
    def forward(self, x): return self.drop(self.res(self.conv(x)))

class Net(nn.Module):
    def __init__(self, b=32, nc=NC, d=0.2):
        super().__init__()
        self.e1 = Enc(1,   b,      0)
        self.e2 = Enc(b,   b*2,  d*0.3)
        self.e3 = Enc(b*2, b*4,  d*0.6)
        self.e4 = Enc(b*4, b*8,  d*0.8)
        self.bn = nn.Sequential(Enc(b*8, b*16, d), nn.Dropout2d(d))
        self.pool = nn.MaxPool2d(2)
        self.u4 = nn.ConvTranspose2d(b*16, b*8, 2, 2)
        self.d4 = Enc(b*16, b*8, d*0.4)
        self.u3 = nn.ConvTranspose2d(b*8,  b*4, 2, 2)
        self.d3 = Enc(b*8,  b*4, d*0.2)
        self.u2 = nn.ConvTranspose2d(b*4,  b*2, 2, 2)
        self.d2 = Enc(b*4,  b*2, 0)
        self.u1 = nn.ConvTranspose2d(b*2,  b,   2, 2)
        self.d1 = Enc(b*2,  b,   0)
        # Deep supervision heads
        self.s3 = nn.Conv2d(b*4, nc, 1)
        self.s2 = nn.Conv2d(b*2, nc, 1)
        self.out= nn.Conv2d(b,   nc, 1)
        # Auxiliary head for rare class emphasis
        self.aux = nn.Sequential(
            nn.Conv2d(b, b, 3, 1, 1, bias=False), nn.BatchNorm2d(b),
            nn.ReLU(True), nn.Conv2d(b, nc, 1))
    def forward(self, x):
        sz = x.shape[2:]
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        d  = self.bn(self.pool(e4))
        d  = self.d4(torch.cat([self.u4(d), e4], 1))
        d  = self.d3(torch.cat([self.u3(d), e3], 1))
        o3 = F.interpolate(self.s3(d), sz, mode='bilinear', align_corners=False)
        d  = self.d2(torch.cat([self.u2(d), e2], 1))
        o2 = F.interpolate(self.s2(d), sz, mode='bilinear', align_corners=False)
        d  = self.d1(torch.cat([self.u1(d), e1], 1))
        if self.training:
            return self.out(d), o2, o3, self.aux(d)
        return self.out(d)

# ── LOSSES ──────────────────────────────────────────────────────
def dice_w(lg, tg, sm=1e-6):
    B,C,H,W = lg.shape
    s = F.softmax(lg, 1)
    o = F.one_hot(tg.clamp(0,C-1), C).permute(0,3,1,2).float()
    p = s[:,1:].reshape(B,C-1,-1)
    t = o[:,1:].reshape(B,C-1,-1)
    inter = (p*t).sum(-1); union = p.sum(-1)+t.sum(-1)
    mask  = (t.sum(-1) > 0).float()
    w = CW[1:].to(lg.device).view(1,C-1)
    return 1.0 - ((2*inter+sm)/(union+sm) * mask * w).sum() / (mask*w).sum().clamp(min=1)

def focal(lg, tg, g=2.0):
    ce = F.cross_entropy(lg, tg.clamp(0,NC-1), reduction='none')
    return ((1 - torch.exp(-ce))**g * ce).mean()

def boundary(lg, tg):
    s = F.softmax(lg, 1)
    o = F.one_hot(tg.clamp(0,NC-1), NC).permute(0,3,1,2).float()
    b = (F.max_pool2d(o[:,1:], 3, stride=1, padding=1) - o[:,1:]).clamp(0,1)
    w = CW[1:].to(lg.device).view(1,-1,1,1)
    return (b * (1-s[:,1:]) * w).sum() / ((b*w).sum() + 1e-6)

def compound(lg, tg):
    tc = tg.clamp(0, NC-1)
    return (F.cross_entropy(lg, tc, label_smoothing=0.02)
            + dice_w(lg, tc)
            + 0.3 * focal(lg, tc)
            + 0.15 * boundary(lg, tc))

def total_loss(outs, tg):
    o1, o2, o3, ax = outs
    main = compound(o1,tg) + 0.3*compound(o2,tg) + 0.15*compound(o3,tg)
    tc   = tg.clamp(0, NC-1)
    rm   = sum((tc==c).float() for c in RARE).clamp(0,1)
    aux_loss = (F.cross_entropy(ax, tc, reduction='none') * (1 + 5*rm)).mean()
    return main + 0.3 * aux_loss

# ── METRICS: pure GPU, no CPU loop ──────────────────────────────
@torch.no_grad()
def fast_dice(lg, tg):
    """GPU-only dice. ~200x faster than per-sample CPU loops."""
    B = lg.shape[0]
    pred = lg.argmax(1)          # (B,H,W) stays on GPU
    sm = 1e-6; D = defaultdict(list)
    for c in range(1, NC):
        p = (pred == c).float().view(B, -1)
        t = (tg   == c).float().view(B, -1)
        active = (t.sum(1) > 0) | (p.sum(1) > 0)
        if not active.any(): continue
        tp  = (p * t).sum(1)[active]
        den = (p.sum(1) + t.sum(1))[active]
        D[c].extend(((2*tp+sm)/(den+sm)).cpu().tolist())
    all_d = [v for vs in D.values() for v in vs]
    return D, float(np.mean(all_d)) if all_d else 0.0

# ── DATASET SPLITS ──────────────────────────────────────────────
def get_splits():
    df = pd.read_csv(OVERVIEW)
    tr, va = [], []; seen = set()
    for name in df['new_file_name'].tolist():
        if not name.endswith('_t2') or 'SPACE' in name: continue
        pid = name.replace('_t2', '')
        if pid in seen: continue
        seen.add(pid)
        sub = df.loc[df['new_file_name']==name, 'subset'].values
        (va if len(sub) and sub[0]=='validation' else tr).append(pid)
    return tr, va

# ── LR SCHEDULE WITH WARMUP ─────────────────────────────────────
def get_lr(ep, start_ep):
    """Linear warmup then cosine decay."""
    rel = ep - start_ep + 1
    if rel <= WARMUP_EP:
        return LR * rel / WARMUP_EP
    t = (rel - WARMUP_EP) / max(EPOCHS - WARMUP_EP, 1)
    return LR_MIN + 0.5 * (LR - LR_MIN) * (1 + np.cos(np.pi * t))

# ── MAIN ────────────────────────────────────────────────────────
device = torch.device('cuda')
print('=' * 62)
print(f'  ATM-Net++ | GPU: {torch.cuda.get_device_name(0)}')
print(f'  {IMG_SIZE}×{IMG_SIZE} | BS={BATCH_SIZE}×{ACCUM}=eff{BATCH_SIZE*ACCUM} | AMP=ON | TTA=ON')
print('=' * 62)

tr_pids, va_pids = get_splits()
found = sum(1 for p in tr_pids if (IMAGES_DIR/f'{p}_t2.mha').exists())
print(f'\nPatients: {len(tr_pids)} train | {len(va_pids)} val | Files found: {found}')
assert found > 0, 'No files found! Re-run Cell 4'

print('\nBuilding data cache...')
ti, tm, tr_rare = build_cache(tr_pids, 'train')
vi, vm, va_rare = build_cache(va_pids, 'val')
ram = (ti.nbytes + tm.nbytes + vi.nbytes + vm.nbytes) // 1024**2
rare_n = sum(1 for r in tr_rare if r > 0.5)
print(f'RAM: ~{ram}MB | Rare slices: {rare_n}/{len(tr_rare)} (10× oversampled)\n')

# WeightedRandomSampler: rare classes get 10× more batches
sampler = WeightedRandomSampler(torch.tensor(tr_rare), len(tr_rare), replacement=True)
tr_dl = DataLoader(DS(ti,tm,Aug()),  batch_size=BATCH_SIZE, sampler=sampler,
                   num_workers=4, pin_memory=True, persistent_workers=True)
va_dl = DataLoader(DS(vi,vm),        batch_size=BATCH_SIZE, shuffle=False,
                   num_workers=4, pin_memory=True, persistent_workers=True)

model = Net(b=32, nc=NC, d=0.2).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model : {n_params/1e6:.2f}M params | ResUNet+CBAM+DeepSup+Aux')
print(f'Loader: {len(tr_dl)} train batches | {len(va_dl)} val batches')

# Resume from checkpoint if it exists on Drive
start_ep = 1; best = 0.0
if CKPT_BEST.exists():
    # Check last_model too — use whichever has higher epoch
    ck_b = torch.load(str(CKPT_BEST), map_location=device)
    ep_b = ck_b.get('epoch', 0)
    ck_use = ck_b
    if CKPT_LAST.exists():
        ck_l = torch.load(str(CKPT_LAST), map_location=device)
        if ck_l.get('epoch', 0) > ep_b:
            ck_use = ck_l
            print(f'Using last_model (ep {ck_l["epoch"]} > best ep {ep_b})')
    model.load_state_dict(ck_use['model_state_dict'], strict=False)
    best     = ck_b.get('best_dice', 0.0)  # always track from best
    start_ep = ck_use.get('epoch', 0) + 1
    print(f'\n✓ Resumed: epoch {ck_use["epoch"]} | best={best:.4f} | continuing from ep {start_ep}')
else:
    print('\nStarting fresh training...')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scaler    = GradScaler()
no_imp    = 0
t0_total  = time.time()

print(f'\n{"Ep":>4}  {"TrLoss":>8}  {"VaDice":>8}  {"Best":>8}  {"Gap":>6}  {"LR":>8}  {"Sec":>5}')
print('─' * 62)

for ep in range(start_ep, EPOCHS+1):
    # Update LR (warmup + cosine)
    lr_now = get_lr(ep, start_ep)
    for pg in optimizer.param_groups: pg['lr'] = lr_now

    # ── TRAIN ──
    model.train(); losses = []; t0 = time.time()
    optimizer.zero_grad(set_to_none=True)
    for step, (imgs, msks) in enumerate(tr_dl):
        imgs = imgs.to(device, non_blocking=True)
        msks = msks.to(device, non_blocking=True)
        with autocast():
            outs = model(imgs)
            loss = total_loss(outs, msks) / ACCUM
        scaler.scale(loss).backward()
        if (step+1) % ACCUM == 0 or (step+1) == len(tr_dl):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        losses.append(loss.item() * ACCUM)
    tr_loss = float(np.mean(losses))
    ep_sec  = time.time() - t0

    # ── VALIDATE with TTA (horizontal flip) ──
    model.eval(); Dc = defaultdict(list)
    with torch.no_grad():
        for imgs, msks in va_dl:
            imgs = imgs.to(device, non_blocking=True)
            msks = msks.to(device, non_blocking=True)
            with autocast():
                # TTA: average original + horizontally flipped prediction
                p1 = F.softmax(model(imgs), 1)
                p2 = F.softmax(model(torch.flip(imgs, [-1])), 1)
                avg = (p1 + torch.flip(p2, [-1])) / 2
            D, _ = fast_dice(avg, msks)
            for c, v in D.items(): Dc[c].extend(v)

    all_v = [v for vs in Dc.values() for v in vs]
    vd    = float(np.mean(all_v)) if all_v else 0.0

    # Estimate train dice from last few batches (light, no extra forward pass)
    with torch.no_grad():
        model.eval()
        imgs_s, msks_s = next(iter(tr_dl))
        imgs_s = imgs_s.to(device); msks_s = msks_s.to(device)
        with autocast(): out_s = model(imgs_s)
        _, td = fast_dice(out_s, msks_s)
    gap = td - vd

    # ── SAVE ──
    if vd > best:
        best = vd; no_imp = 0
        pc = {CN[c]: float(np.mean(v)) for c,v in Dc.items() if v}
        torch.save({'epoch': ep, 'model_state_dict': model.state_dict(),
                    'best_dice': best, 'per_class_dice': pc,
                    'cfg': {'img_size': IMG_SIZE, 'nc': NC}}, CKPT_BEST)
    else:
        no_imp += 1

    # Save every 5 epochs regardless (crash recovery)
    if ep % 5 == 0:
        torch.save({'epoch': ep, 'model_state_dict': model.state_dict(),
                    'best_dice': best, 'cfg': {'img_size': IMG_SIZE, 'nc': NC}},
                   CKPT_LAST)

    # ── LOG ──
    flag = '  ★' if vd == best else ''
    print(f'{ep:>4}  {tr_loss:>8.4f}  {vd:>8.4f}  {best:>8.4f}  '
          f'{gap:>+6.3f}  {lr_now:>8.2e}  {ep_sec:>4.0f}s{flag}')

    if vd >= 0.90:
        print('\n🎯 TARGET ACHIEVED: Dice ≥ 0.90!'); break
    if vd >= 0.85:
        print(f'  📈 Dice {vd:.4f} — past 0.85!')
    if no_imp >= PATIENCE:
        print(f'\n⏸ Early stop — no improvement for {PATIENCE} epochs'); break

    gc.collect(); torch.cuda.empty_cache()

t_total = (time.time() - t0_total) / 3600
print('─' * 62)
print(f'\n✓ Training complete: {t_total:.2f}h | Best Dice: {best:.4f}')
print(f'✓ Checkpoint: {CKPT_BEST}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6: Final Evaluation — per-class Dice + IoU + TTA
# Run after training to get full breakdown
# ═══════════════════════════════════════════════════════════════
import torch, torch.nn.functional as F
import numpy as np, json
from pathlib import Path
from collections import defaultdict

CKPT_BEST = Path('/content/drive/MyDrive/ATM_Net_outputs/best_model.pth')
assert CKPT_BEST.exists(), 'No checkpoint — run training first'

ck = torch.load(str(CKPT_BEST), map_location='cpu')
print('=' * 62)
print(f'  ATM-Net++ FINAL EVALUATION')
print(f'  Epoch: {ck["epoch"]} | Best Dice: {ck["best_dice"]:.4f}')
print('=' * 62)

# Load model in eval mode
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# (Net class must still be defined from Cell 5)
model_e = Net(b=32, nc=NC, d=0.2).to(device)
model_e.load_state_dict(ck['model_state_dict'], strict=False)
model_e.eval()
print(f'✓ Model loaded to {device}')

aD = defaultdict(list); aI = defaultdict(list); n_sl = 0; sm = 1e-6
VERT = list(range(1,9)); IVD = list(range(10,18))

with torch.no_grad():
    for imgs, msks in va_dl:
        imgs = imgs.to(device); msks = msks.to(device)
        with autocast():
            p1 = F.softmax(model_e(imgs), 1)
            p2 = F.softmax(model_e(torch.flip(imgs,[-1])), 1)
            probs = (p1 + torch.flip(p2,[-1])) / 2
        pred_np = probs.argmax(1).cpu().numpy()
        gt_np   = msks.cpu().numpy()
        n_sl += pred_np.shape[0]
        for b in range(pred_np.shape[0]):
            for c in range(1, NC):
                p = (pred_np[b]==c).astype(float).ravel()
                t = (gt_np[b]==c).astype(float).ravel()
                if t.sum()==0 and p.sum()==0: continue
                tp = (p*t).sum(); fp = (p*(1-t)).sum(); fn = ((1-p)*t).sum()
                aD[c].append((2*tp+sm)/(2*tp+fp+fn+sm))
                aI[c].append((tp+sm)/(tp+fp+fn+sm))

print(f'\n  {"Class":<22} {"Dice":>8}  {"IoU":>8}  {"N":>6}')
print('  ' + '─'*46)
all_d=[]; all_i=[]; vert_d=[]; ivd_d=[]; res={}
for c in range(1, NC):
    if not aD[c]: continue
    d = float(np.mean(aD[c])); io = float(np.mean(aI[c]))
    nm = CN[c]; res[nm] = {'dice':d,'iou':io,'n':len(aD[c])}
    all_d.append(d); all_i.append(io)
    if c in VERT: vert_d.append(d)
    if c in IVD:  ivd_d.append(d)
    tag = '  ★' if d>=0.90 else '  ✓' if d>=0.80 else '  ·' if d>=0.70 else '  +' if d>=0.60 else ''
    print(f'  {nm:<22} {d:>8.4f}  {io:>8.4f}  {len(aD[c]):>6}{tag}')

md = float(np.mean(all_d)) if all_d else 0
mi = float(np.mean(all_i)) if all_i else 0
vm = float(np.mean(vert_d)) if vert_d else 0
im = float(np.mean(ivd_d))  if ivd_d  else 0
print('  ' + '─'*46)
print(f'  {"MEAN FOREGROUND":<22} {md:>8.4f}  {mi:>8.4f}')
print(f"""
  ╔══════════════════════════════════════════╗
  ║  Vertebrae Dice : {vm:.4f}                 ║
  ║  IVD Dice       : {im:.4f}                 ║
  ║  Mean Dice(TTA) : {md:.4f}                 ║
  ║  Mean IoU (TTA) : {mi:.4f}                 ║
  ║  Val slices     : {n_sl}                   ║
  ╚══════════════════════════════════════════╝
""")
status = ('🎯 TARGET ACHIEVED ≥ 0.90!' if md>=0.90 else
          '✅ GOAL MET ≥ 0.85!'        if md>=0.85 else
          '📈 Close — continue training' if md>=0.80 else
          '📊 Good — more epochs needed')
print(f'  Status: {status}')

final = {'mean_dice':md,'mean_iou':mi,'vertebrae_dice':vm,'ivd_dice':im,
         'total_val_slices':n_sl,'per_class':res}
result_file = Path('/content/drive/MyDrive/ATM_Net_outputs/evaluation_results.json')
with open(result_file,'w') as f: json.dump(final,f,indent=2)
print(f'\n✓ Results saved → {result_file}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7: Download checkpoint to local machine
# ═══════════════════════════════════════════════════════════════
from google.colab import files
from pathlib import Path
import shutil

CKPT_BEST = Path('/content/drive/MyDrive/ATM_Net_outputs/best_model.pth')
RESULT    = Path('/content/drive/MyDrive/ATM_Net_outputs/evaluation_results.json')

if CKPT_BEST.exists():
    # Copy to /content for download
    shutil.copy(str(CKPT_BEST), '/content/best_model.pth')
    files.download('/content/best_model.pth')
    print('✓ Downloading best_model.pth...')
    print('  Save to: outputs/gpu_run/best_model.pth on your local machine')
    print('  Then restart server.py to use the new model')
else:
    print('No checkpoint yet — run training first')

if RESULT.exists():
    shutil.copy(str(RESULT), '/content/evaluation_results.json')
    files.download('/content/evaluation_results.json')
    print('✓ Downloading evaluation_results.json...')